<a href="https://colab.research.google.com/github/jceltruda/Projects-in-AI-and-ML/blob/main/Project_6/Task_3/03_training_and_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/SimRec')

Mounted at /content/drive


# SimRec Training & Evaluation

This notebook provides the full training and evaluation loop for the SimRec model.

## 1. Setup

In [ ]:
!pip install -q torch numpy tqdm pandas

In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd
from types import SimpleNamespace

# Import all SimRec components from the module generated by notebook 2
from simrec_module import (
    SimRec,
    WarpSampler,
    data_partition,
    create_similarity_distirbution,
    LinearScheduleWithWarmup,
    NoneSchedule,
    evaluate_test,
    evaluate_valid,
    evaluate_train,
    PAD_IDX,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


## 2. Configuration

All hyperparameters are set here. To switch datasets, change `DATASET_NAME` and the paths, and optionally adjust hyperparameters using the presets in the comments.

In [ ]:
# Dataset paths
DATASET_NAME = "Beauty"
EMBEDDING_MODEL_SAFE = "thenlper_gte-large"

DATA_DIR = DATASET_NAME
DATASET_PATH = os.path.join(DATA_DIR, f"{DATASET_NAME}.txt")
ITEM_FREQ_PATH = os.path.join(DATA_DIR, f"{DATASET_NAME}-train_item_freq.txt")
SIMILARITY_INDICES_PATH = os.path.join(DATA_DIR, f"{DATASET_NAME}-similarity-indices-{EMBEDDING_MODEL_SAFE}.pt")
SIMILARITY_VALUES_PATH = os.path.join(DATA_DIR, f"{DATASET_NAME}-similarity-values-{EMBEDDING_MODEL_SAFE}.pt")

# Hyperparameters: Beauty defaults
args = SimpleNamespace(
    # Data
    dataset=DATASET_PATH,
    item_frequency=ITEM_FREQ_PATH,
    similarity_indices=SIMILARITY_INDICES_PATH,
    similarity_values=SIMILARITY_VALUES_PATH,

    # Similarity
    similarity_threshold=0.9,
    temperature=0.5,

    # Lambda scheduling
    lambd=0.3,
    lambd_scheduling="LINEAR",       # "LINEAR" or "NONE"
    lambd_warmup_steps=1000,
    lambd_steps=81000,

    # Model architecture
    hidden_units=100,
    num_blocks=3,
    num_heads=1,
    dropout_rate=0.5,
    maxlen=50,

    # Training
    batch_size=128,
    lr=0.0001,
    l2_emb=0.0,
    num_epochs=210,
    device=device,

    # Checkpoints
    train_dir=f"results/{DATASET_NAME}",
    inference_only=True,
    state_dict_path='results/Beauty/SimRec.epoch=210.pth',            # Set to a .pth path to resume training
)

# Preset overrides for other datasets
# # Tools
# args.hidden_units = 100; args.num_blocks = 3; args.dropout_rate = 0.5
# args.lr = 0.0001; args.temperature = 0.5; args.lambd = 0.3
# args.similarity_threshold = 0.9; args.num_epochs = 210

# # HomeKitchen
# args.hidden_units = 100; args.num_blocks = 3; args.dropout_rate = 0.5
# args.lr = 0.0001; args.temperature = 0.5; args.lambd = 0.3
# args.similarity_threshold = 0.9; args.num_epochs = 210

# # PetSupplies
# args.hidden_units = 100; args.num_blocks = 3; args.dropout_rate = 0.5
# args.lr = 0.0001; args.temperature = 0.5; args.lambd = 0.3
# args.similarity_threshold = 0.9; args.num_epochs = 210

print("Configuration:")
for k, v in sorted(vars(args).items()):
    print(f"  {k}: {v}")

Configuration:
  batch_size: 128
  dataset: Beauty/Beauty.txt
  device: cuda
  dropout_rate: 0.5
  hidden_units: 100
  inference_only: True
  item_frequency: Beauty/Beauty-train_item_freq.txt
  l2_emb: 0.0
  lambd: 0.3
  lambd_scheduling: LINEAR
  lambd_steps: 81000
  lambd_warmup_steps: 1000
  lr: 0.0001
  maxlen: 50
  num_blocks: 3
  num_epochs: 210
  num_heads: 1
  similarity_indices: Beauty/Beauty-similarity-indices-thenlper_gte-large.pt
  similarity_threshold: 0.9
  similarity_values: Beauty/Beauty-similarity-values-thenlper_gte-large.pt
  state_dict_path: results/Beauty/SimRec.epoch=210.pth
  temperature: 0.5
  train_dir: results/Beauty


## 3. Load Data

In [ ]:
# Load item frequencies
item_freq_df = pd.read_csv(args.item_frequency, delimiter=' ', names=['id', 'freq'])
item_freq_df['id'] += 1  # id 0 is reserved for PAD
item_freq = pd.Series(item_freq_df.freq.values, index=item_freq_df.id).to_dict()

# Partition data into train / valid / test
dataset = data_partition(args.dataset)
[user_train, user_valid, user_test, usernum, itemnum] = dataset

print(f"Users: {usernum}, Items: {itemnum}")
cc = sum(len(user_train[u]) for u in user_train)
print(f"Average sequence length: {cc / len(user_train):.2f}")

Users: 52133, Items: 57226
Average sequence length: 5.63


In [ ]:
# Load precomputed similarity matrices
similarity_indices = torch.load(args.similarity_indices, map_location=args.device)
similarity_values = torch.load(args.similarity_values, map_location=args.device)

# Apply similarity threshold
if args.similarity_threshold < 1:
    similarity_values[similarity_values <= args.similarity_threshold] = -float('inf')
else:
    # Make the self-similarity maximal (baseline mode)
    similarity_indices = torch.arange(similarity_indices.shape[0], device=args.device).reshape(-1, 1)
    similarity_values = torch.ones_like(similarity_indices, dtype=torch.float)

# Handle padding index (0) by increasing index values
similarity_indices += 1

# Prepend a row for the padding index
similarity_indices = torch.cat([
    torch.arange(similarity_indices.shape[1], device=args.device).unsqueeze(dim=0),
    similarity_indices
], dim=0)
similarity_values = torch.cat([
    torch.full((1, similarity_values.shape[1]), fill_value=-float('inf'), device=args.device),
    similarity_values
], dim=0)

print(f"Similarity indices shape: {similarity_indices.shape}")
print(f"Similarity values shape:  {similarity_values.shape}")

Similarity indices shape: torch.Size([57227, 1000])
Similarity values shape:  torch.Size([57227, 1000])


## 4. Initialize Model

In [ ]:
# Create output directory
os.makedirs(args.train_dir, exist_ok=True)

# Save config
with open(os.path.join(args.train_dir, 'args.txt'), 'w') as f:
    f.write('\n'.join([f"{k},{v}" for k, v in sorted(vars(args).items())]))

# Compute training steps for lambda scheduler
num_batch = len(user_train) // args.batch_size
training_steps = num_batch * args.num_epochs
print(f"{training_steps} training steps in total ({num_batch} batches/epoch x {args.num_epochs} epochs)")

# Lambda scheduler
if args.lambd_scheduling == 'LINEAR':
    lambda_scheduler = LinearScheduleWithWarmup(args.lambd, args.lambd_warmup_steps, min(training_steps, args.lambd_steps))
else:
    lambda_scheduler = NoneSchedule(args.lambd)

# Data sampler
sampler = WarpSampler(user_train, usernum, itemnum,
                      batch_size=args.batch_size, maxlen=args.maxlen, n_workers=3)

# Model
model = SimRec(usernum, itemnum, args).to(args.device)
model_total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"SimRec model: {model_total_params:,} trainable parameters")

# Optionally load a checkpoint
epoch_start_idx = 1
if args.state_dict_path is not None:
    try:
        model.load_state_dict(torch.load(args.state_dict_path, map_location=torch.device(args.device)))
        tail = args.state_dict_path[args.state_dict_path.find('epoch=') + 6:]
        epoch_start_idx = int(tail[:tail.find('.')]) + 1
        print(f"Loaded checkpoint, resuming from epoch {epoch_start_idx}")
    except Exception as e:
        print(f"Failed loading state dict: {e}")
        print(f"Path: {args.state_dict_path}")

85470 training steps in total (407 batches/epoch x 210 epochs)
SimRec model: 5,910,900 trainable parameters
Loaded checkpoint, resuming from epoch 211


## 5. Inference-Only Mode

Skip training and evaluate a saved checkpoint. Set `args.inference_only = True` and
`args.state_dict_path` above to use this cell.

In [ ]:
if args.inference_only:
    model.eval()
    print("Running inference-only evaluation (5 rounds)...")
    test_eval_results = [evaluate_test(model, dataset, args) for _ in range(5)]

    test_hr_list = [e[0][1] for e in test_eval_results]
    test_ndcg_list = [e[0][0] for e in test_eval_results]

    test_hr = np.array(test_hr_list).mean()
    test_ndcg = np.array(test_ndcg_list).mean()
    print(f"\nTest Results: NDCG@10: {test_ndcg:.4f}, HR@10: {test_hr:.4f}")
else:
    print("Inference-only mode is OFF. Proceed to the training cell below.")

Running inference-only evaluation (5 rounds)...
................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................
Test Results — NDCG@10: 0.4129, HR@10: 0.5818


## 6. Training Loop

In [ ]:
if not args.inference_only:
    # Loss functions and optimizer
    bce_criterion = torch.nn.BCEWithLogitsLoss()
    cross_entropy_criterion = torch.nn.CrossEntropyLoss()
    adam_optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, betas=(0.9, 0.98))

    # Logging
    f_log = open(os.path.join(args.train_dir, 'log.txt'), 'w')
    fname = f'SimRec.epoch={args.num_epochs}.pth'

    T = 0.0
    t0 = time.time()
    best_val_hr = -float('inf')
    best_results = {}
    global_step = 0

    model.train()

    for epoch in range(epoch_start_idx, args.num_epochs + 1):
        for step in range(num_batch):
            lambd = lambda_scheduler.get_lambd()
            u, seq, pos, neg = sampler.next_batch()
            u, seq, pos, neg = np.array(u), np.array(seq), np.array(pos), np.array(neg)

            pos_logits, neg_logits, logits = model(u, seq, pos, neg)
            pos_labels = torch.ones(pos_logits.shape, device=args.device)
            neg_labels = torch.zeros(neg_logits.shape, device=args.device)

            adam_optimizer.zero_grad()
            indices = np.where(pos != PAD_IDX)
            loss = bce_criterion(pos_logits[indices], pos_labels[indices])
            loss += bce_criterion(neg_logits[indices], neg_labels[indices])

            # Similarity distillation loss
            pos_t = torch.tensor(pos, device=args.device, dtype=torch.long).view(-1)
            pad_indices = pos_t == PAD_IDX
            pos_t = pos_t[~pad_indices]
            logits_flat = logits.view(-1, itemnum + 1)
            logits_flat = logits_flat[~pad_indices]

            targets_distribution = create_similarity_distirbution(
                similarity_indices, similarity_values, args.temperature, pos_t
            )
            loss = lambd * cross_entropy_criterion(logits_flat / args.temperature, targets_distribution) + (1 - lambd) * loss

            # Optional L2 regularization on embeddings
            if args.l2_emb > 0:
                for param in model.item_emb.parameters():
                    loss += args.l2_emb * torch.norm(param)

            loss.backward()
            adam_optimizer.step()
            lambda_scheduler.step()
            global_step += 1

        print(f"loss in epoch {epoch}: {loss.item():.6f}")

        # Periodic evaluation
        if epoch % 20 == 0 or epoch == args.num_epochs:
            model.eval()
            t1 = time.time() - t0
            T += t1
            print('Evaluating', end='')

            test_eval_results = [evaluate_test(model, dataset, args) for _ in range(5)]
            val_eval_results = [evaluate_valid(model, dataset, args) for _ in range(5)]

            test_id_hr = test_eval_results[-1][-2]
            test_id_ndcg = test_eval_results[-1][-1]
            valid_id_hr = val_eval_results[-1][-2]
            valid_id_ndcg = val_eval_results[-1][-1]

            test_hr = np.array([e[0][1] for e in test_eval_results]).mean()
            test_ndcg = np.array([e[0][0] for e in test_eval_results]).mean()
            val_hr = np.array([e[0][1] for e in val_eval_results]).mean()
            val_ndcg = np.array([e[0][0] for e in val_eval_results]).mean()

            (train_ndcg, train_hr), train_id_hr, train_id_ndcg = evaluate_train(model, dataset, args)

            s = (f'\nepoch:{epoch}, time: {T:.1f}s, '
                 f'train (NDCG@10: {train_ndcg:.4f}, HR@10: {train_hr:.4f}), '
                 f'valid (NDCG@10: {val_ndcg:.4f}, HR@10: {val_hr:.4f}), '
                 f'test (NDCG@10: {test_ndcg:.4f}, HR@10: {test_hr:.4f})')
            print(s)
            f_log.write(s + '\n')
            f_log.flush()

            # Save best checkpoint
            if val_hr > best_val_hr:
                torch.save(model.state_dict(), os.path.join(args.train_dir, fname))
                torch.save(test_id_hr, os.path.join(args.train_dir, "test_id_hr.pt"))
                torch.save(test_id_ndcg, os.path.join(args.train_dir, "test_id_ndcg.pt"))
                torch.save(valid_id_hr, os.path.join(args.train_dir, "valid_id_hr.pt"))
                torch.save(valid_id_ndcg, os.path.join(args.train_dir, "valid_id_ndcg.pt"))
                torch.save(train_id_hr, os.path.join(args.train_dir, "train_id_hr.pt"))
                torch.save(train_id_ndcg, os.path.join(args.train_dir, "train_id_ndcg.pt"))
                print(f"New best val HR@10: {val_hr:.4f}. Saving checkpoint.")
                best_val_hr = val_hr
                best_results = {
                    "test/best_NDCG@10": test_ndcg,
                    "test/best_HR@10": test_hr,
                    "val/best_NDCG@10": val_ndcg,
                    "val/best_HR@10": val_hr,
                    "train/best_NDCG@10": train_ndcg,
                    "train/best_HR@10": train_hr,
                }

            t0 = time.time()
            model.train()

    f_log.close()
    sampler.close()
    print("\nTraining complete!")

loss in epoch 1: 2.396109
loss in epoch 2: 3.448959
loss in epoch 3: 4.005294
loss in epoch 4: 3.979331
loss in epoch 5: 3.923400
loss in epoch 6: 3.929605
loss in epoch 7: 3.767606
loss in epoch 8: 3.739799
loss in epoch 9: 3.853065
loss in epoch 10: 3.729688
loss in epoch 11: 3.741959
loss in epoch 12: 3.664932
loss in epoch 13: 3.629378
loss in epoch 14: 3.643175
loss in epoch 15: 3.604470
loss in epoch 16: 3.421660
loss in epoch 17: 3.426325
loss in epoch 18: 3.494977
loss in epoch 19: 3.481456
loss in epoch 20: 3.268409
Evaluating...........................................................................................................................................................................................................................................................................................................................................................................................................................................................................

## 7. Final Results

In [ ]:
if not args.inference_only and best_results:
    print("=" * 60)
    print("Best Results (selected by validation HR@10)")
    print("=" * 60)
    for metric, value in sorted(best_results.items()):
        print(f"  {metric}: {value:.4f}")
    print(f"\nCheckpoint saved to: {os.path.join(args.train_dir, fname)}")
    print(f"Training log saved to: {os.path.join(args.train_dir, 'log.txt')}")

## 8. Load & Evaluate Best Checkpoint

After training completes, load the best saved checkpoint and run a final evaluation.

In [ ]:
# Load best checkpoint
fname = f'SimRec.epoch={args.num_epochs}.pth'
best_checkpoint_path = os.path.join(args.train_dir, fname)
if os.path.exists(best_checkpoint_path):
    eval_model = SimRec(usernum, itemnum, args).to(args.device)
    eval_model.load_state_dict(torch.load(best_checkpoint_path, map_location=args.device))
    eval_model.eval()

    print("Evaluating best checkpoint (5 rounds)...")
    final_test_results = [evaluate_test(eval_model, dataset, args) for _ in range(5)]

    final_test_hr = np.array([e[0][1] for e in final_test_results]).mean()
    final_test_ndcg = np.array([e[0][0] for e in final_test_results]).mean()

    print(f"\nFinal Test Results (best checkpoint):")
    print(f"  NDCG@10: {final_test_ndcg:.4f}")
    print(f"  HR@10:   {final_test_hr:.4f}")

    del eval_model
else:
    print(f"No checkpoint found at {best_checkpoint_path}")

Evaluating best checkpoint (5 rounds)...
................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................
Final Test Results (best checkpoint):
  NDCG@10: 0.4116
  HR@10:   0.5800
